# ДЗ-1. Ваш участок ПРАЙМ

Вы — аналитик команды подписки ПРАЙМ. У каждого аналитика свой **участок**: одна таблица, один сегмент и одно окно времени. Это тот самый персональный срез из `prime.assignments`, с которым вы работали на первом семинаре, и больше он ни у кого не повторяется.

В понедельник руководитель просит сводку по участку — восемь задач. В каждой две части: **расчёт** (функция на Python) и **вопрос**, на который вы отвечаете словами в ячейке «✍️ Ответ на вопрос N». Числа у всех разные, поэтому и выводы у каждого свои.

### Как устроен ноутбук

| Часть | Что там | Что делаете вы |
|---|---|---|
| **1. Подготовка** | готовый код: подключение к базе и выгрузка вашего участка | вписываете пять значений в ячейку 1.1 и запускаете ячейки по порядку |
| **2. Восемь задач** | условия, заготовки функций, вопросы | пишете код функций и ответы словами |
| **3. Итог** | готовый код: сводка ваших ответов и проверка их формата | запускаете перед сдачей |
| **4. Исследование** | две открытые задачи для тех, кто хочет 9 или 10 | по желанию |

**Оценка.** Части 1–3 — это оценка до 8 баллов: сделали всё аккуратно, выводы верные — 8, и это «отлично». 9 и 10 — только за исследование в части 4. Все критерии — в [тексте задания](https://github.com/tikhomirovd/python-for-ba-hse-2026/tree/master/задания/дз-1-окружение-и-репозиторий), там же — как сдавать.

Запускайте ячейки сверху вниз (`Shift + Enter`). Перед сдачей — **Kernel → Restart Kernel and Run All Cells**: ноутбук должен пройти целиком без красного.

Ноутбук лежит в **вашем** репозитории как `notebooks/hw1.ipynb`, а JupyterLab запускается из корня `prime-monitor` командой `uv run jupyter lab`.

## Часть 1. Подготовка — готовый код

Код в этой части писать не нужно — он готов. Единственная ваша правка — пять значений в ячейке 1.1. Номер ячейки — это номер заголовка над ней: «ячейка 1.1» — код сразу под заголовком «1.1 Ваш участок».

Код длинный, потому что аккуратно обходит несколько ловушек, до которых мы дойдём позже. Прочитать его полезно, разбирать каждую строку не обязательно.

| Ячейка | Что делает | Что делаете вы |
|---|---|---|
| 1.1 Ваш участок | хранит пять значений участка | вписываете значения |
| 1.2 Подключение | подключается к базе через ваш `.env` | запускаете |
| 1.3 Строка участка | показывает вашу строку из `prime.assignments` | запускаете, копируете значения в 1.1 и запускаете 1.1 ещё раз |
| 1.4 Запрос | описывает, как достать участок из любой из четырёх таблиц | запускаете |
| 1.5 Выгрузка | достаёт участок в переменную `rows` | запускаете — дальше вы работаете с `rows` |
| 1.6 Формат ответов | служебная функция для итога | запускаете |

### 1.1 Ваш участок

Впишите значения из своей строки `prime.assignments` — строками, в кавычках. Какие они, покажет ячейка 1.3: запустите 1.2 и 1.3, скопируйте напечатанные строки сюда и **запустите эту ячейку ещё раз**, иначе Python продолжит видеть старые значения.

Это **единственное место**, где указан участок. При проверке преподаватель подставит сюда другой участок — и все ответы должны пересчитаться сами. Поэтому ячейку не удаляйте и не пересоздавайте, а значения больше нигде не повторяйте.

In [ ]:
TABLE = 'transactions'
SEGMENT_COLUMN = 'channel'
SEGMENT = 'app'
PERIOD_START = '2025-04-01'
PERIOD_END = '2025-04-30'

### 1.2 Подключение

Подключаемся к учебной базе и заводим функцию `query(sql)`: она отправляет SQL-запрос и возвращает результат списком словарей — одна строка результата, один словарь, как на семинарах. Пароля в ноутбуке нет: его знает только ваш `.env`.

Если здесь ошибка `Пакет prime не найден` — ноутбук запущен не в окружении вашего репозитория. Если `Не найдена переменная PRIME_DSN` — нет файла `.env` в корне `prime-monitor`.

Соединение открывается на каждый запрос и сразу закрывается: так после сна ноутбука или смены Wi-Fi ничего не отваливается, а общий сервер не держит лишних подключений.

In [ ]:
import sys
from collections import Counter, defaultdict  # понадобятся в задачах
from datetime import date
from decimal import Decimal

from sqlalchemy import text

# get_engine() живёт в вашем репозитории, в src/prime/config.py.
# Она читает строку подключения из .env — поэтому пароля здесь нет.
try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()


def query(sql: str, **params) -> list[dict]:
    """Выполнить SQL-запрос и вернуть строки списком словарей {колонка: значение}."""
    try:
        with engine.connect() as conn:
            # Моменты времени в базе хранятся с часовым поясом. День события считаем
            # по UTC — так он у всех одинаковый, какие бы настройки ни стояли у вас.
            conn.execute(text("set time zone 'UTC'"))
            result = conn.execute(text(sql), params)
            return [dict(row) for row in result.mappings()]
    finally:
        # Соединение не держим: сервер у всей группы общий, а после сна ноутбука
        # старое соединение всё равно мёртвое.
        engine.dispose()


print("подключились к базе как:", query("select current_user as login")[0]["login"])

### 1.3 Строка участка

Показывает вашу строку из `prime.assignments` — ту же, что на первом семинаре. Чужих строк база не показывает. Скопируйте напечатанное в ячейку 1.1 и запустите 1.1 ещё раз.

За то, что в 1.1 стоят именно ваши значения, отвечаете вы: код с чужим участком спокойно всё посчитает, только не то.

In [ ]:
mine = query("select * from prime.assignments where login = current_user")

if mine:
    row = mine[0]
    print("Ваш участок. Скопируйте в ячейку 1.1 и запустите её:\n")
    print(f"TABLE = {row['table_name']!r}")
    print(f"SEGMENT_COLUMN = {row['segment_column']!r}")
    print(f"SEGMENT = {row['segment']!r}")
    print(f"PERIOD_START = {str(row['period_start'])!r}")
    print(f"PERIOD_END = {str(row['period_end'])!r}")
else:
    # Так бывает, когда ноутбук запускает преподаватель под своей ролью.
    print("Строки участка для этого пользователя в базе нет.")

### 1.4 Запрос

Четыре таблицы устроены по-разному, но для задач нужно одно и то же. Поэтому запрос приводит любую из них к **пяти полям**:

| Поле | Что это |
|---|---|
| `row_id` | идентификатор строки |
| `actor_id` | участник — тот, чьи это строки |
| `day` | день события, тип `datetime.date` |
| `ok` | состоялось ли событие: `True` или `False` |
| `value` | сколько: деньги, минуты или часы, тип `Decimal` |

Что это значит в **вашей** таблице:

| Таблица | Одна строка — это | Участник | Сегмент | `ok = True` | `ok = False` | `value` |
|---|---|---|---|---|---|---|
| `payments` | попытка списать плату за подписку | подписка | способ оплаты: `card`, `sbp`, `wallet`, `balance` | списание прошло | не прошло или деньги вернули | сумма списания, ₽ |
| `transactions` | покупка клиента в партнёрском сервисе | клиент | канал: `app` — приложение, `web` — сайт, `pos` — касса | покупка прошла | отклонена или проведена и отменена | сумма покупки, ₽; бывает ноль и меньше нуля |
| `service_usage` | сессия в сервисе ПРАЙМ | клиент | устройство | сессия записана корректно: конец позже начала | конец не позже начала | длительность, минуты |
| `support_tickets` | обращение в поддержку | клиент | тема обращения | обращение решено | ещё в работе | время от создания до решения, часы; у нерешённых — `None` |

**Участник** здесь — просто тот, чьи это строки. Не путайте с участниками подписки из `subscription_members`, о которых шла речь на первом семинаре. А у `support_tickets` сумма `value` по дню растёт и от числа обращений, и от медленных решений — это не одно и то же.

Как работает запрос:

1. Блок `acts` выбирает **1000 участников** вашего сегмента за ваше окно. Это выборка, а не весь сегмент: поэтому строк в `rows` меньше, чем вы насчитали в хвосте первого семинара, — там был весь срез. Подгонять под то число не нужно. В ответах словами пишите «в выгрузке» или «среди 1000 подписок / клиентов», а не «по всем картам за квартал». Перемешивание через `md5` делает выбор похожим на случайный, но при каждом запуске — одинаковым.
2. Основной запрос берёт строки этих участников — **только в вашем сегменте и в вашем окне** (тот же фильтр, что в `acts`) — и переименовывает колонки в пять полей.
3. В `{фигурных скобках}` — названия колонок из словаря `COLUMNS`: они подставляются под вашу таблицу. После двоеточия (`:segment`, `:period_start`, `:period_end`) — ваши значения из ячейки 1.1.
4. Граница окна: момент события `>=` первого дня и `<` дня, следующего за последним. Так последний день попадает в окно целиком.

In [ ]:
# Как в каждой из четырёх таблиц называются колонки, из которых получаются пять полей.
COLUMNS = {
    "payments": {
        "segment_column": "payment_method",   # колонка сегмента
        "row_id": "payment_id",                # -> row_id
        "actor": "subscription_id",            # -> actor_id
        "moment": "paid_at",                   # момент события -> day
        "ok": "status = 'success'",            # -> ok
        "value": "amount",                     # -> value
    },
    "transactions": {
        "segment_column": "channel",
        "row_id": "transaction_id",
        "actor": "client_id",
        "moment": "occurred_at",
        "ok": "status = 'success'",
        "value": "amount",
    },
    "service_usage": {
        "segment_column": "device_type",
        "row_id": "usage_id",
        "actor": "client_id",
        "moment": "started_at",
        "ok": "ended_at > started_at",
        "value": "extract(epoch from (ended_at - started_at)) / 60.0",      # секунды -> минуты
    },
    "support_tickets": {
        "segment_column": "category",
        "row_id": "ticket_id",
        "actor": "client_id",
        "moment": "created_at",
        "ok": "resolved_at is not null",
        "value": "extract(epoch from (resolved_at - created_at)) / 3600.0",  # секунды -> часы
    },
}

SLICE_SQL = """
with acts as (
    select actor_id
    from (
        select distinct {actor}::text as actor_id
        from prime.{table}
        where {segment_column} = :segment
          and {moment} >= cast(:period_start as date)
          and {moment} <  cast(:period_end as date) + 1
    ) s
    order by md5(actor_id)
    limit 1000
)
select t.{row_id}::text as row_id,
       t.{actor}::text  as actor_id,
       t.{moment}::date as day,
       {ok}             as ok,
       {value}          as value
from prime.{table} t
join acts a on a.actor_id = t.{actor}::text
where t.{segment_column} = :segment
  and t.{moment} >= cast(:period_start as date)
  and t.{moment} <  cast(:period_end as date) + 1
order by t.{row_id}
"""

print("запрос описан")

### 1.5 Выгрузка

Достаём участок из базы в переменную `rows` — список словарей, по одному на строку. **Все задачи ниже работают с `rows`.**

Сколько в выгрузке строк, участников и дней — ячейка не печатает: это вы посчитаете сами в задаче 1.

In [ ]:
# В текст запроса подставляются только известные названия таблиц и колонок —
# поэтому сначала проверяем, что значения в ячейке 1.1 из этого списка.
if TABLE not in COLUMNS:
    raise ValueError(f"TABLE должна быть одной из {list(COLUMNS)}, сейчас {TABLE!r}")
if SEGMENT_COLUMN != COLUMNS[TABLE]["segment_column"]:
    raise ValueError(f"у таблицы {TABLE} колонка сегмента называется {COLUMNS[TABLE]['segment_column']!r}")
try:
    window_days = (date.fromisoformat(PERIOD_END) - date.fromisoformat(PERIOD_START)).days + 1
except (TypeError, ValueError):
    raise ValueError("PERIOD_START и PERIOD_END — строки вида '2025-04-01'") from None

# Собираем запрос под вашу таблицу и выполняем. Обычно несколько секунд;
# если висит дольше минуты — см. «Если что-то не работает» в тексте задания.
slice_sql = SLICE_SQL.format(table=TABLE, **COLUMNS[TABLE])
rows = query(slice_sql, segment=SEGMENT, period_start=PERIOD_START, period_end=PERIOD_END)

# Защита от ошибок в значениях 1.1: пустая выгрузка или дни за пределами окна.
if not rows:
    raise ValueError("выгрузка пустая — проверьте значения в ячейке 1.1")
if len({row["day"] for row in rows}) > window_days:
    raise ValueError("в выгрузке дней больше, чем в окне — граница периода съехала")

print("Выгрузка готова. Так выглядят первые три строки:")
rows[:3]

### 1.6 Формат ответов

Служебная функция: по ней итог в части 3 проверяет, что ответ записан в нужном формате — список, число, дата строкой. **Верно ли посчитано, она не знает и не сообщает** — это проверяет преподаватель. Код ячейки свёрнут — так задумано, просто запустите её.

In [ ]:
# Какой формат ответа ожидается в каждой задаче — человеческими словами.
EXPECTED = {
    1: "list из 4 элементов: [int, int, float до 2 знаков, int]",
    2: "int",
    3: "dict: ключи из Mon…Sun, значения float до 2 знаков",
    4: "list [str, int]",
    5: "list из 3 элементов, каждый — list ['ГГГГ-ММ-ДД', float до 2 знаков]",
    6: "list [int, 'ГГГГ-ММ-ДД']",
    7: "list [float до 2 знаков, int]",
    8: "float до 2 знаков",
}


def format_problem(n: int, answer) -> str | None:
    """Что не так с ФОРМАТОМ ответа задачи n (не с правильностью). None — всё в порядке."""

    def money(x) -> bool:  # float, у которого не больше двух знаков после запятой
        return type(x) is float and round(x, 2) == x

    def iso_day(x) -> bool:  # дата строкой 'ГГГГ-ММ-ДД'
        try:
            return type(x) is str and date.fromisoformat(x).isoformat() == x
        except ValueError:
            return False

    if answer is None:
        return "не решена: функция вернула None — в ней остался ... или забыт return"
    checks = {
        1: lambda a: type(a) is list and len(a) == 4 and type(a[0]) is int and type(a[1]) is int
                     and money(a[2]) and type(a[3]) is int,
        2: lambda a: type(a) is int,
        3: lambda a: type(a) is dict and set(a) <= {"Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"}
                     and all(money(v) for v in a.values()),
        4: lambda a: type(a) is list and len(a) == 2 and type(a[0]) is str and type(a[1]) is int,
        5: lambda a: type(a) is list and len(a) == 3
                     and all(type(p) is list and len(p) == 2 and iso_day(p[0]) and money(p[1]) for p in a),
        6: lambda a: type(a) is list and len(a) == 2 and type(a[0]) is int and iso_day(a[1]),
        7: lambda a: type(a) is list and len(a) == 2 and money(a[0]) and type(a[1]) is int,
        8: lambda a: money(a),
    }
    if checks[n](answer):
        return None
    return f"нужен {EXPECTED[n]}, сейчас {answer!r}"


print("проверка формата готова")

## Часть 2. Восемь задач

У каждой задачи два шага:

1. **Посчитать.** Допишите функцию `taskN(rows)` — она должна вернуть ответ ровно в указанном формате. Имена функций и переменных `answer_N` не меняйте: по ним работа проверяется автоматически, в том числе на другом участке.
2. **Ответить на вопрос.** Для ответа почти всегда нужно досчитать одно-два числа — для этого под вопросом есть пустая ячейка. Сам ответ пишется в ячейке «✍️ Ответ на вопрос N»: дважды щёлкните по ней, **заголовок не трогайте**, замените строку курсивом своим текстом и нажмите `Shift + Enter`.

**Ячейки расчётов тоже выполняются на контрольном участке.** Считайте в них от `rows` и `answer_N`; конкретные даты, `actor_id` и сегмент руками не вписывайте — на другом участке их нет, и ячейка упадёт.

**Не получилась задача — не оставляйте падающий код.** Ячейка с ошибкой останавливает Run All, и ноутбук не проходит целиком — это минус баллы за воспроизводимость. Верните в тело функции `...`: она вернёт `None`, итог покажет «❌ не решена», а остальное проверится как обычно.

**Про ответы словами.** Код можно писать вместе с ИИ-ассистентом — это нормально, отметьте это в декларации. А ответы пишите сами: здесь оценивается, как думаете вы. **Текст, написанный LLM, получает 0 баллов.** Хороший ответ — 2–4 предложения, в них ваши числа и ваш вывод. Сомнительный вывод баллов не приносит, даже если числа верные.

**Ноутбук не скажет, правильно ли вы посчитали.** Итог в части 3 проверяет только формат. Правильность проверяет преподаватель — на вашем участке и на контрольном.

### Три особенности данных, на которых спотыкаются чаще всего

* **`value` приходит типом `Decimal`, а не `float`.** Это точный тип для денег. `Decimal` складывается с `Decimal` и с целыми числами, а с `float` — нет: `Decimal("1.5") + 0.5` падает с `TypeError`. В ноутбуке С2-идиомы суммы копились от `total = 0.0` — здесь так не выйдет. Копите суммы в `Decimal` (`defaultdict(Decimal)`, `sum(...)` без стартового `0.0`), а к `float` приводите в самом конце: `float(round(сумма, 2))`.
* **В `support_tickets` у нерешённых обращений `value` равно `None`** — времени до решения ещё нет. Проверяйте `row["value"] is not None` там, где складываете `value`, а не `if row["value"]`: второе молча выкинет и честный ноль. Сами строки с `None` из `rows` не удаляйте — это нерешённые обращения, и они нужны в задачах 1, 2 и в вопросе 1.
* **Строки с `ok = True` и отрицательным или нулевым `value` не выбрасывайте.** В `transactions` такие есть — например, покупка на ноль рублей. Считайте их как есть.

Нужные приёмы разобраны в [ноутбуке С2-идиомы](https://github.com/tikhomirovd/python-for-ba-hse-2026/blob/master/02-среда-git-python/семинар/С2-идиомы.ipynb): генераторные выражения, множества, `defaultdict` и `Counter`, `sorted` с ключом, `enumerate`, `try/except`.

### Задача 1. Паспорт участка

Руководитель начинает каждую неделю с одного вопроса: «Сколько у нас всего — и сколько из этого настоящего?» Прежде чем что-то анализировать, надо знать масштаб участка.

**Что посчитать**

Четыре числа по вашей выгрузке `rows`:

1. сколько всего строк;
2. сколько строк с `ok = True` — событий, которые состоялись;
3. сколько `value` набежало по строкам с `ok = True` — выручка, минуты или часы, смотря какая у вас таблица;
4. сколько **различных** дней встречается в выгрузке — по всем строкам, а не только по состоявшимся.

**Формат ответа:** `[всего, с_ok, сумма, дней]` — например `[2222, 2100, 123456.7, 17]`: целое, целое, `float` до двух знаков, целое.

**Подсказка:** `len(rows)`; `sum(1 for row in rows if ...)`; множество дней `{row["day"] for row in rows}`.

In [ ]:
def task1(rows: list[dict]) -> list:
    total_rows = len(rows)
    ok_rows = sum(1 for row in rows if row["ok"])
    total_value = sum(
        row["value"]
        for row in rows
        if row["ok"] and row["value"] is not None)

    total_value = float(round(total_value, 2))

    days = len({row["day"] for row in rows})

    return [total_rows, ok_rows, total_value, days]
answer_1 = task1(rows)
answer_1

**Вопрос 1.** **Что не случилось.** Какая доля строк вашей выгрузки не состоялась (`ok = False`)? Что это значит в жизни вашей таблицы — что именно не произошло? Сколько `value` приходится на эти строки — или объясните, почему у вашей таблицы это число не имеет смысла или его нельзя посчитать.

In [ ]:
# Дополнительные расчёты для ответа на вопрос 1

failed_rows = [
    row
    for row in rows
    if row["ok"] is False
]

failed_count = len(failed_rows)

failed_share = round(
    failed_count / len(rows) * 100,
    1
)

failed_value = sum(
    row["value"]
    for row in failed_rows
    if row["value"] is not None
)


true_actors = {
    row["actor_id"]
    for row in rows
    if row["ok"] is True
}

false_actors = {
    row["actor_id"]
    for row in rows
    if row["ok"] is False
}

only_false_count = len(false_actors - true_actors)
mixed_count = len(true_actors & false_actors)


failed_by_day = Counter(
    row["day"]
    for row in failed_rows
)

if failed_by_day:
    peak_failed_day, peak_failed_count = max(
        failed_by_day.items(),
        key=lambda item: (item[1], item[0])
    )
else:
    peak_failed_day = None
    peak_failed_count = 0


print("неуспешных строк:", failed_count)
print("доля неуспешных строк, %:", failed_share)
print(
    "value по неуспешным строкам:",
    round(failed_value, 2)
)
print(
    "участников только с False:",
    only_false_count
)
print(
    "участников с обоими исходами:",
    mixed_count
)
print(
    "больше всего неуспешных строк за день:",
    peak_failed_count
)
print(
    "дата максимума неуспешных строк:",
    peak_failed_day
)


# В rows хранится только дата day.
# Поэтому точные интервалы в минутах считаем
# в отдельной проверочной выгрузке repeat_rows.

repeat_sql = SLICE_SQL.replace(
    "t.{moment}::date as day,",
    "t.{moment} as moment,\n"
    "       t.{moment}::date as day,"
).format(
    table=TABLE,
    **COLUMNS[TABLE]
)

repeat_rows = query(
    repeat_sql,
    segment=SEGMENT,
    period_start=PERIOD_START,
    period_end=PERIOD_END
)


repeat_by_actor = defaultdict(list)

for row in repeat_rows:
    repeat_by_actor[row["actor_id"]].append(row)


for actor_rows in repeat_by_actor.values():
    actor_rows.sort(
        key=lambda row: row["moment"]
    )


followups = []

for failed_row in (
    row
    for row in repeat_rows
    if row["ok"] is False
):
    next_success = next(
        (
            row
            for row in repeat_by_actor[
                failed_row["actor_id"]
            ]
            if row["ok"] is True
            and row["moment"] > failed_row["moment"]
        ),
        None
    )

    if next_success is not None:
        followups.append(
            (failed_row, next_success)
        )


intervals_minutes = [
    (
        success["moment"] - failed["moment"]
    ).total_seconds() / 60
    for failed, success in followups
]


if intervals_minutes:
    average_interval = (
        sum(intervals_minutes)
        / len(intervals_minutes)
    )
    minimum_interval = min(intervals_minutes)
else:
    average_interval = None
    minimum_interval = None


same_day_followups = sum(
    success["moment"].date()
    == failed["moment"].date()
    for failed, success in followups
)

same_amount_followups = sum(
    success["value"] == failed["value"]
    for failed, success in followups
)


print(
    "неуспешных операций с последующей успешной:",
    len(followups)
)
print(
    "без последующей успешной операции:",
    failed_count - len(followups)
)
print(
    "средний интервал до следующей успешной, минут:",
    round(average_interval, 2)
    if average_interval is not None
    else None
)
print(
    "минимальный интервал до следующей успешной, минут:",
    round(minimum_interval, 2)
    if minimum_interval is not None
    else None
)
print(
    "последующих успешных в тот же день:",
    same_day_followups
)
print(
    "последующих успешных на ту же сумму:",
    same_amount_followups
)


#### ✍️ Ответ на вопрос 1


В выборке было 1 551 операция по 1 000 клиентам. Из них 1 494 операции, или 96,3%, завершились успешно, а 57 операций, или 3,7%, — нет. В бизнес-смысле это означает, что 57 попыток клиентов совершить покупку не привли к состоявшейся операции: покупка могла быть отклонена или впоследствии отменена. Сумма  по этим строкам составила 84 703,53 ₽, однако это не обязательно потерянная выручка, поскольку деньги могли не списаться или вернуться клиенту. У 26 клиентов все операции были неуспешными — в этой выборке у них не было ни одной состоявшейся покупки. Ещё 30 клиентов имели и успешные и неуспешные операции
ДОПОЛНИТЕЛЬНО (интересно чек):
После 19 неуспешных операций у того же клиента нашлась последующая успешная операция. Среднее время между ними составило 9 146,73 минуты, то есть примерно 6 дней 8 часов 27 минут. Самый короткий интервал составил 775,75 минуты — около 12 часов 56 минут. При этом ни в одном случае сумма следующей успешной операции не совпала с суммой неуспешной, поэтому нельзя сказать, что это были повторныуе попытки совершить ту же покупку. У остальных 38 неуспешных операций последующей успешной операции в пределах выбранного периода не нашлось. Больше всего неуспешных операций было 11 апр — 6 случаев

### Задача 2. И получилось, и нет

Самые интересные участники — те, у кого в одном окне было и «получилось», и «не получилось»: оплата сначала не прошла, а потом прошла; одну покупку отклонили, другую провели. Руководитель хочет знать, сколько таких.

**Что посчитать**

Сколько **различных** участников (`actor_id`) имеют в выгрузке хотя бы одну строку с `ok = True` **и** хотя бы одну строку с `ok = False`.

**Формат ответа:** целое число, например `123`. Ноль — тоже законный ответ.

**Подсказка:** два множества участников и операция `&` между ними.

In [ ]:
def task2(rows: list[dict]) -> int:
    true_actors= {row["actor_id"]
                       for row in rows
                       if row["ok"] is True}
    false_actors = {row["actor_id"]
                      for row in rows
                       if row["ok"] is False}
    both = true_actors & false_actors
    both_number = len(both)
    return both_number
answer_2 = task2(rows)
answer_2

**Вопрос 2.** **Почему столько.** Сколько в среднем строк приходится на одного участника в вашей выгрузке? Объясните, почему ответ задачи 2 получился именно таким. Если это ноль или единица — это не ошибка: объясните, откуда он берётся. Подумайте, к чему относится сегмент: к отдельной строке (у одного участника строки могут быть с разными значениями) или к участнику целиком (у всех его строк оно одно).

In [ ]:
# Участники с успешными и неуспешными строками
true_actors = {
    row["actor_id"]
    for row in rows
    if row["ok"] is True
}

false_actors = {
    row["actor_id"]
    for row in rows
    if row["ok"] is False
}

both = true_actors & false_actors

# Все уникальные участники
actors = {
    row["actor_id"]
    for row in rows
}

# Группируем строки по участникам
rows_by_actor = defaultdict(list)

for row in rows:
    rows_by_actor[row["actor_id"]].append(row)


# Сколько строк приходится на одного участника
average_rows = round(
    len(rows) / len(actors),
    2
)

# Разделяем участников по сочетанию статусов
only_true = true_actors - false_actors
only_false = false_actors - true_actors
# Сколько строк приходится на клиентов,
# у которых были оба результата
both_row_counts = [
    len(rows_by_actor[actor_id])
    for actor_id in both
]

# Распределение: сколько клиентов имеют
# 1 строку, 2 строки, 3 строки и так далее
rows_count_by_actor = defaultdict(int)

for actor_id, actor_rows in rows_by_actor.items():
    rows_count_by_actor[len(actor_rows)] += 1


print("всего строк:", len(rows))
print("уникальных участников:", len(actors))
print("среднее число строк на участника:", average_rows)

print()
print("ответ задачи 2:", len(both))
print("участников только с True:", len(only_true))
print("участников только с False:", len(only_false))
print("участников и с True, и с False:", len(both))

if len(both) > 0:
    print()
    print(
        "среднее число строк у участников "
        "с обоими статусами:",
        round(sum(both_row_counts) / len(both_row_counts), 2)
    )
    print(
        "минимальное число строк у них:",
        min(both_row_counts)
    )
    print(
        "максимальное число строк у них:",
        max(both_row_counts)
    )

print()
print("распределение участников по числу строк:")

for rows_number in sorted(rows_count_by_actor):
    print(
        rows_number,
        "строк:",
        rows_count_by_actor[rows_number],
        "участников"
    )
    

#### ✍️ Ответ на вопрос 2
В  выгрузке было 1 551 транзакций и 1 000 уникалдьных клиентов. Поэтому среднее число строк на одного клиента составило 1551 / 1000 = 1,55.
Ответ задачи 2 равен 30, потому что у 30 разных клиентов встретились оба результата: хотя бы одна успешная и хотя бы одна неуспешная операция. Это не 30 строк, а именно 30 уникальных Actor_id. У такого клиента должно быть минимум две строки, поэтому среднее число строк у этих клиентов получилось равным 3, а количество строк у них варьируется от 2 до 7.
Всего 679 клиентов имели только одну операцию, а у 321 клиента было несколько операций. Не у всех клиентов с повторными операциями менялся результат: у большинства несколько строк были либо только успешными, либо только неуспешными. Именно поэтому из 1 000 клиентов в пересечение двух множеств попали только 30.
Сегмент channel относится к отдельной транзакции, а не к клиенту целиком. В этой выгрузке мы отобрали операции выбранного канала, но это не означает, что клиент пользуется только им: у него могут быть операции в других каналах, которые просто не попали в текущую выборку.


### Задача 3. Главный день недели

Команда решает, в какой день недели ставить дежурство и запускать рассылки. Нужна раскладка участка по дням недели.

**Что посчитать**

Сумма `value` по строкам с `ok = True` — отдельно для каждого дня недели.

**Формат ответа:** словарь `{'Mon': 12345.67, 'Tue': ...}` — только дни недели, которые встретились; суммы `float` до двух знаков. Название дня берите из списка `WEEKDAYS` по номеру `row["day"].weekday()` (понедельник — 0).

**Подсказка:** `defaultdict(Decimal)`. Почему не `day.strftime('%a')`: результат зависит от языковых настроек — стоит где-нибудь вызвать `locale.setlocale`, и вместо `'Tue'` получится `'вт'`, а ответ разойдётся с форматом.

In [ ]:
WEEKDAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]  # WEEKDAYS[0] — понедельник


def task3(rows: list[dict]) -> dict:
    sums = defaultdict(Decimal)
    for row in rows:
        if row["ok"] is True:
            weekday = WEEKDAYS[row["day"].weekday()]
            sums[weekday] +=  row["value"]
    return {
    day: float(round(total, 2))
    for day, total in sums.items()
}
answer_3 = task3(rows)
answer_3

**Вопрос 3.** **Сумма или среднее.** Сколько **различных дат** вашей выгрузки приходится на каждый день недели? Считайте даты, а не строки. Какой день лидирует по сумме из задачи 3, а какой — по средней сумме на одну такую дату (сумма из задачи 3, делённая на число дат этого дня недели)? Если лидеры разные — какой ответ вы понесли бы руководителю и почему; если одинаковые — объясните, почему так вышло.

In [ ]:
from collections import defaultdict


# Собираем разные даты по каждому дню недели
dates_by_weekday = defaultdict(set)

for row in rows:
    if row["ok"] is True:
        day_name = WEEKDAYS[row["day"].weekday()]
        dates_by_weekday[day_name].add(row["day"])


# Сколько разных дат приходится на каждый день недели
date_counts = {
    day: len(dates)
    for day, dates in dates_by_weekday.items()
}


# Средняя сумма на одну дату
average_by_day = {
    day: answer_3[day] / date_counts[day]
    for day in answer_3
}


# Лидер по общей сумме
sum_leader = max(
    answer_3,
    key=answer_3.get
)


# Лидер по средней сумме на одну дату
average_leader = max(
    average_by_day,
    key=average_by_day.get
)


print("количество разных дат по дням недели:")
print(date_counts)

print()
print("средняя сумма на одну дату:")
print({
    day: round(value, 2)
    for day, value in average_by_day.items()
})

print()
print("лидер по общей сумме:", sum_leader)
print("лидер по средней сумме на дату:", average_leader)

if sum_leader == average_leader:
    print("лидеры совпадают")
else:
    print("лидеры разные")

#### ✍️ Ответ на вопрос 3

В  выборке на пн, чт, пт, сб, вс пришлось по 4 даты, а на вт и ср — по 5 дат. По общей сумме лидирует вт — 382 819,52 ₽. Он же лидирует по средней сумме на одну дату — 76 563,90 ₽, поэтому руководителю я бы назвала вторник самым результативным днём. Это не связано только с тем, что вторников было больше: даже в расчёте на одну дату вторник опережает остальные дни.

### Задача 4. Самый активный участник

Маркетинг хочет наградить самого активного участника участка — того, у кого больше всего состоявшихся событий. Приз уже куплен.

**Что посчитать**

Участник с наибольшим числом строк с `ok = True` и это число. Если таких несколько — тот, чей `actor_id` меньше при сравнении как текст (так сравнивает `min`).

**Формат ответа:** `[actor_id, число строк]` — например `['aaaaaaaa-1111-2222-3333-bbbbbbbbbbbb', 9]`: текст (`str`) и целое.

**Подсказка:** `Counter` по строкам с `ok = True` считает, сколько состоявшихся событий у каждого участника; `max` находит наибольшее число, `min` — наименьший `actor_id` среди тех, у кого оно такое.

In [ ]:
def task4(rows: list[dict]) -> list:
    counts = Counter(
        row["actor_id"]
        for row in rows
        if row["ok"] is True
    )

    max_count = max(counts.values())

    winners = [
        actor_id
        for actor_id, count in counts.items()
        if count == max_count
    ]

    # Если лидеров несколько, выбираем минимальный actor_id как текст
    return [min(winners), max_count]


answer_4 = task4(rows)
answer_4

**Вопрос 4.** **Кто победил на самом деле.** Сколько участников делят первое место? Если больше одного — что на самом деле выбрало вашего «победителя»: данные или правило? И что в вашей таблице вообще значит «самый активный» — стоит ли маркетингу вручать приз по такому рейтингу?

In [ ]:
from collections import Counter


counts = Counter(
    row["actor_id"]
    for row in rows
    if row["ok"] is True
)


# Уникальные значения количества успешных транзакций
# от большего к меньшему
place_values = sorted(
    set(counts.values()),
    reverse=True
)


for place, transaction_number in enumerate(
    place_values[:3],
    start=1
):
    actors_on_place = [
        actor_id
        for actor_id, count in counts.items()
        if count == transaction_number
    ]

    print(place, "место")
    print(
        "участников делят место:",
        len(actors_on_place)
    )
    print(
        "успешных транзакций у каждого:",
        transaction_number
    )
    print(
        "участники:",
        actors_on_place
    )
    print()

#### ✍️ Ответ на вопрос 4

В выборке первое место занял один клиент, его id - 682ad418-8c1d-4f0e-9f60-6e98b5b779f5, у него 11 успешных транзакций. Второе и третье места также не делят несколько участников: на них находятся клиенты с 10 и 9 успешными транзакциями соответственно (то есть первое место не аутлаер. не ош данных, а реальный человек, сумма покупок которого не сильно отличается от других). Поэтому победителя выбрали сами данные, а не дополнительное правило для разрешения ничьей. В данном рейтинге «самый активный» означает клиента с наибольшим числом состоявшихся покупок, поэтому маркетинг может использовать такой показатель для награды за частоту покупок, но он не показывает общую ценность клиента и сумму его покупок.

### Задача 5. Три пиковых дня

Чтобы планировать нагрузку на команду и серверы, нужно знать пиковые дни окна.

**Что посчитать**

Три даты с наибольшей суммой `value` по строкам с `ok = True` — по убыванию суммы. Если суммы равны, раньше идёт более ранняя дата.

**Формат ответа:** `[['2024-03-15', 98765.43], ['2024-03-02', 91200.1], ['2024-03-21', 90000.0]]` — дата строкой `'ГГГГ-ММ-ДД'`, сумма `float` до двух знаков.

**Подсказка:** сначала словарь «день → сумма», потом `sorted(..., key=...)`: ключ-кортеж `(-сумма, день)` сортирует по убыванию суммы, а при равенстве — по возрастанию даты. Дата строкой — `day.isoformat()`.

In [ ]:
def task5(rows: list[dict]) -> list:
    sums_by_day = defaultdict(Decimal)

    for row in rows:
        if row["ok"] is True:
            sums_by_day[row["day"]] += row["value"]

    sorted_days = sorted(
        sums_by_day.items(),
        key=lambda item: (-item[1], item[0])
    )

    return [
        [
            day.isoformat(),
            float(round(total, 2))
        ]
        for day, total in sorted_days[:3]
    ]


answer_5 = task5(rows)
answer_5

**Вопрос 5.** **Откуда пик.** Какие это дни недели, сколько среди них суббот и воскресений? Сформулируйте одну гипотезу, почему пик пришёлся на эти даты, и напишите, какой расчёт на данных её подтвердил бы или опроверг.

In [ ]:
from datetime import date


peak_dates = [
    date.fromisoformat(point[0])
    for point in answer_5
]

peak_weekdays = [
    WEEKDAYS[day.weekday()]
    for day in peak_dates
]

weekend_peaks = sum(
    1
    for day in peak_dates
    if day.weekday() >= 5
)


print("три пиковые даты:", answer_5)
print("дни недели пиков:", peak_weekdays)
print(
    "пиковых дат на субботу и воскресенье:",
    weekend_peaks
)

#### ✍️ Ответ на вопрос 5

Три самые крупные суммы пришлись на 15 апреля — вт, 22 апреля — вт и 14 апреля — пн. Среди пиковых дат нет суббот и воскресений. Возможная причина — более высокая активность клиентов в начале рабочей недели, особенно по вторникам. Проверить эту гипотезу можно, сравнив среднюю сумму успешных операций за одну дату по дням недели, а не только общие суммы: если вторник и рабочие дни в среднем будут лидировать - гипотеза подтвердится.

### Задача 6. Когда набралась половина

Финансам важно, равномерно ли набирается сумма внутри окна: к какому дню набегает половина суммы по состоявшимся событиям.

**Что посчитать**

1. Возьмите дни, в которых есть хотя бы одна строка с `ok = True`, и расставьте по возрастанию даты.
2. Идите по ним и копите сумму `value` по строкам с `ok = True`.
3. Найдите первый день, на котором накопленное **достигло половины** суммы из пункта 3 задачи 1 (всё `value` по строкам с `ok = True`) или превысило её, и его порядковый номер в списке дней из шага 1, считая с 1.

**Формат ответа:** `[номер, 'ГГГГ-ММ-ДД']` — например `[12, '2024-03-15']`.

**Подсказка:** `sums` — словарь «день → сумма», как в задаче 5. `for number, day in enumerate(sorted(sums), start=1):` — `enumerate` сам считает номер.

In [ ]:
def task6(rows: list[dict]) -> list:
    # Сумма успешных операций по каждой дате
    sums_by_day = defaultdict(Decimal)

    for row in rows:
        if row["ok"] is True:
            sums_by_day[row["day"]] += row["value"]

    # Общая сумма успешных операций
    total = Decimal("0")

    for total_by_day in sums_by_day.values():
        total += total_by_day

    half = total / 2
    accumulated = Decimal("0")

    # Идём по датам по порядку
    for number, day in enumerate(
        sorted(sums_by_day),
        start=1
    ):
        accumulated += sums_by_day[day]

        if accumulated >= half:
            return [
                number,
                day.isoformat()
            ]


answer_6 = task6(rows)
answer_6

**Вопрос 6.** **Равномерно ли.** Какую долю дней заняла первая половина суммы: номер дня из задачи 6, делённый на число дней в списке из шага 1? Номер дня целый, поэтому при равномерном накоплении получилось бы около половины, округлённой вверх (15 из 30, 16 из 31) — сравнивайте с этим, а не с ровными 50 %. У вас раньше или позже — и что это говорит о том, как менялся участок внутри окна? Если вышло у середины — так и напишите.

In [ ]:
active_days = {
    row["day"]
    for row in rows
    if row["ok"] is True
}

half_share = round(
    100 * answer_6[0] / len(active_days),
    1
)

uniform_day = (len(active_days) + 1) // 2

uniform_reference = round(
    100 * uniform_day / len(active_days),
    1
)


print("дней с успешными операциями:", len(active_days))
print("половина суммы достигнута на дне №:", answer_6[0])
print("дата первой половины суммы:", answer_6[1])
print("доля дней до первой половины суммы, %:", half_share)
print(
    "ориентир при равномерном накоплении, %:",
    uniform_reference
)


if answer_6[0] < uniform_day:
    print("половина суммы набралась раньше середины периода")
elif answer_6[0] > uniform_day:
    print("половина суммы набралась позже середины периода")
else:
    print("половина суммы набралась примерно в середине периода")

#### ✍️ Ответ на вопрос 6

В моей выгрузке (участок) успешные операции были во все 30 дней. Половина общей суммы накопилась к 16му дню — 16 апреля, то есть за 53,3% периода. При равномерном накоплении данных показатель составит около 50%, поэтому половина суммы набралась немного позже середины периода. Следовательно, во второй половине окна суммы успешных операций были нмного выше.

### Задача 7. Функция, которая не падает

Соседний отдел присылает суммы строками — так, как их набрал оператор: с запятой, с пробелами, иногда с прочерком. Нужна функция, которая превращает такую строку в число и не роняет ежедневный отчёт. Данные в этой задаче у всех одинаковые — список `RAW`.

**Что посчитать**

1. Функция `to_value(x)`: получает строку (или `None`) и возвращает `float` — или `None`, если число разобрать нельзя. Падать нельзя ни на одном значении. Правила записи:
   * точка и запятая — десятичный разделитель: `'89.10'` → `89.1`, `'349,90'` → `349.9`;
   * пробел и неразрывный пробел внутри числа — разделители тысяч: `'1 499,50'` → `1499.5`;
   * пробелы по краям ничего не значат;
   * прочерк `'—'` и пустая строка — не число: `None`, а не `0.0`.
2. По списку `RAW`: сумма всего, что разобралось, и сколько значений отброшено. `None` тоже считается отброшенным.

**Формат ответа:** `[сумма, отброшено]` — `float` до двух знаков и целое.

**Подсказка:** `try: ... except ValueError: ...` — ловите именно `ValueError`, а не всё подряд. `None` проверьте до `try`: `float(None)` бросает `TypeError`, а не `ValueError`. `x.split()` делит строку по любым пробелам, включая неразрывный.

In [ ]:
RAW = ["199.00", "1 499,50", None, "—", "349,90", "2\u00a0100", "", "  89.10  "]
def to_value(x: str | None) -> float | None:
    if x is None:
        return None

    # Убираем пробелы, в том числе неразрывные,
    # и заменяем запятую на точку
    text = "".join(x.split()).replace(",", ".")

    try:
        return float(text)
    except ValueError:
        return None


def task7(raw: list) -> list:
    total = 0.0
    dropped = 0

    for item in raw:
        value = to_value(item)

        if value is None:
            dropped += 1
        else:
            total += value

    return [
        round(total, 2),
        dropped
    ]


answer_7 = task7(RAW)
answer_7

**Вопрос 7.** **Что опаснее.** Представьте ежедневный отчёт о выручке, построенный на этой функции. Что опаснее: функция, которая падает на `'—'`, или функция, которая молча возвращает `None`? Что бы вы добавили в отчёт, чтобы потеря была видна руководителю? Опирайтесь на свой ответ задачи 7: сколько из восьми значений было бы потеряно.

In [ ]:
print("сумма разобранных значений:", answer_7[0])
print("отброшено значений:", answer_7[1])
print("всего значений в RAW:", len(RAW))
print(
    "доля отброшенных, %:",
    round(answer_7[1] / len(RAW) * 100, 1)
)

#### ✍️ Ответ на вопрос 7

В списке было 8 значений, из которых 3, или 37,5%, не удалоось разобрать. Опаснее функция, которая молча возвращает None: отчёт продолжит строиться, но сумма окажется неполной, и ошибка может остаться незамеченной. В данном случае сумма разобранных знчений составила 4 237,5, поэтому в отчёт нужно добавить количество и долю отброшенных значений, а также сами некорректные записи.

### Задача 8. Поправка на день недели

Аналитики из соседней команды прислали коэффициенты дней недели, чтобы сравнивать недели между собой. Прислали только будни — как это обычно и бывает.

**Что посчитать**

1. Для каждого дня недели возьмите сумму `value` по строкам с `ok = True` — как в задаче 3, но **до округления**.
2. Приведите её к `float` и умножьте на коэффициент дня из `WEIGHTS`. Если дня в справочнике нет — коэффициент `1.0`.
3. Сложите все произведения. Округлите до двух знаков **только итог**.

**Формат ответа:** одно число `float` до двух знаков, например `123456.78`.

**Подсказка:** `WEIGHTS.get(day, 1.0)` возвращает коэффициент или `1.0`, если дня нет. `Decimal * float` падает с `TypeError` — сначала `float(...)`.

In [ ]:
WEIGHTS = {"Mon": 1.00, "Tue": 1.05, "Wed": 1.05, "Thu": 1.00, "Fri": 0.95}
def task8(rows: list[dict]) -> float:
    sums = defaultdict(Decimal)

    for row in rows:
        if row["ok"] is True:
            day = WEEKDAYS[row["day"].weekday()]
            sums[day] += row["value"]

    weighted_total = 0.0

    for day, value in sums.items():
        coefficient = WEIGHTS.get(day, 1.0)
        weighted_total += float(value) * coefficient

    return round(weighted_total, 2)


answer_8 = task8(rows)
answer_8

**Вопрос 8.** **Что изменила поправка.** Какие дни недели пошли с коэффициентом по умолчанию? На сколько процентов (с точностью до десятых) взвешенная сумма отличается от обычной суммы из задачи 1 — процент считайте от обычной суммы? В какой ситуации на данных ПРАЙМ такой `.get` с умолчанием спрятал бы ошибку, вместо того чтобы о ней сообщить?

In [ ]:
used_days = {
    WEEKDAYS[row["day"].weekday()]
    for row in rows
    if row["ok"] is True
}

default_days = sorted(
    day
    for day in used_days
    if day not in WEIGHTS
)

difference_share = round(
    100 * (answer_8 - answer_1[2]) / answer_1[2],
    1
)

print("дни с коэффициентом по умолчанию:", default_days)
print("обычная сумма из задачи 1:", answer_1[2])
print("взвешенная сумма:", answer_8)
print(
    "изменение относительно обычной суммы, %:",
    difference_share
)

#### ✍️ Ответ на вопрос 8

Коэффициент по умолчанию применился к субботе и воскресенью, поскольку для них нет отдельных значений в словаре WEIGHTS. Обычная сумма успешных операций составила 1 928 460,8 руб., а взвешенная — 1 949 346,8 руб., то есть она увеличилась на 1,1%. Использование .get может скрыть ошибку, если день недели случайно записать с опечаткой или забыть добавить его в словарь: тогда автоматически применится коэффициент 1,0 вместо сообщения об ошибке.

## Часть 3. Итог — готовый код

При **Restart Kernel and Run All Cells** эта ячейка выполняется после всех задач: перед сдачей посмотрите её вывод. Она показывает ваши ответы и проверяет только формат, но не правильность.

Последняя строка вывода — служебная: по ней преподаватель сверяет ответы автоматически. Ячейку не удаляйте.

In [ ]:
import json


def typed(x):
    """Для служебной строки: всё, кроме list/dict/str/int/float/bool/None, помечаем типом."""
    if type(x) is dict:
        return {str(key): typed(value) for key, value in x.items()}
    if type(x) is list:
        return [typed(value) for value in x]
    if x is None or type(x) in (bool, int, float, str):
        return x
    return f"<{type(x).__name__}> {x!r}"


# Участок из ячейки 1.1 и ответы answer_1 … answer_8; у каждого ответа проверяем формат.
params = {"table_name": TABLE, "segment_column": SEGMENT_COLUMN, "segment": SEGMENT,
          "period_start": PERIOD_START, "period_end": PERIOD_END}
answers = {n: globals().get(f"answer_{n}") for n in range(1, 9)}
formats = {n: format_problem(n, answer) for n, answer in answers.items()}

for n, answer in answers.items():
    print(f"задача {n}: {answer!r}")
    print("   ✅ формат в порядке" if formats[n] is None else f"   ❌ {formats[n]}")

print("\nНе забудьте ответы словами — ячейки «✍️ Ответ на вопрос N» в части 2.")
print("\n--- служебная строка, не удаляйте ---")
print("HW1_ANSWERS=" + json.dumps(
    {"slice": params, "answers": typed(answers), "format": formats}, ensure_ascii=False
))

## Часть 4. Исследование — для тех, кто хочет 9 или 10

**Эта часть необязательна.** Части 1–3 — это оценка до 8 баллов, и 8 — это «отлично». 9 и 10 ставятся только за исследование: два задания, каждое до 1 балла сверху. Засчитываются, если основная часть набрала не меньше 15 баллов из 20.

Здесь нет готового формата ответа и заготовок. Вы сами решаете, что посчитать, считаете это кодом в ячейках ниже и пишете вывод. Ассистент может помочь с кодом, но ответ зависит от того, что вы увидите в своих данных. Текст вывода — ваш: правило про LLM действует и здесь.

| Каждое исследование оценивается так | Балл |
|---|---|
| числа верные и получены кодом в ноутбуке, а не вписаны руками | 0,4 |
| вывод следует из чисел, названо, чего ваш расчёт **не** доказывает | 0,3 |
| код исследования выполняется у преподавателя целиком — в том числе на контрольном участке | 0,2 |
| изложено по схеме: вопрос → как проверяли → что получилось → что это значит | 0,1 |

Код исследования, как и всё остальное, считайте от значений ячейки 1.1: на контрольном участке он должен отработать без правок.

База умеет считать сама: `count(*)`, `count(distinct …)`, `sum(…)`, условие внутри агрегата — `count(*) filter (where …)`, разбивка — `group by`. Весь сегмент в Python не выгружайте: запрос дольше 2 минут сервер оборвёт. Функция `query` и словарь `COLUMNS` из части 1 здесь пригодятся.

### Исследование А. Можно ли верить вашей тысяче?

Руководитель прочитал сводку и спрашивает: «Вы смотрели тысячу участников, а в сегменте их гораздо больше. Насколько ваши числа верны для всего сегмента? Если в следующем месяце value на участника упадёт на 5 %, мы это заметим по такой тысяче?»

**Что сделать**

1. По **всему** вашему сегменту за окно — одним запросом к базе, не выгружая строки — посчитайте: сколько участников, долю строк с `ok = True`, `value` на участника (сумма `value` по строкам с `ok = True`, делённая на число различных участников).
2. Те же показатели — по вашей тысяче, из `rows`.
3. Возьмите **не меньше 20 других тысяч** участников того же сегмента и окна и посчитайте показатели для каждой.
4. Ответьте: насколько ваша тысяча отличается от всего сегмента? Каков разброс между тысячами? Заметно ли по одной тысяче падение `value` на участника на 5 % — и сколько участников нужно брать, чтобы такое падение было заметно?

**Подсказки.** Другая тысяча — это другое перемешивание. Ячейку 1.4 не меняйте — сделайте копию запроса в своей ячейке: `salted_sql = SLICE_SQL.replace("md5(actor_id)", "md5(actor_id || :salt)").format(table=TABLE, **COLUMNS[TABLE])` и вызывайте `query(salted_sql, segment=SEGMENT, period_start=PERIOD_START, period_end=PERIOD_END, salt=str(i))` с разными `i`. Разброс удобно описать стандартным отклонением (`statistics.pstdev`). Как разброс зависит от размера выборки — поищите «стандартная ошибка среднего».

In [ ]:
from math import ceil, sqrt
from statistics import pstdev


def calculate_metrics(sample_rows: list[dict]) -> dict:
    actors = {
        row["actor_id"]
        for row in sample_rows
    }

    total_rows = len(sample_rows)

    successful_rows = sum(
        1
        for row in sample_rows
        if row["ok"] is True
    )

    successful_value = sum(
        row["value"]
        for row in sample_rows
        if row["ok"] is True
        and row["value"] is not None
    )

    return {
        "actors": len(actors),
        "rows": total_rows,
        "ok_share": 100 * successful_rows / total_rows,
        "value_per_actor": float(successful_value) / len(actors)
    }


# Статистика по всему сегменту
ALL_STATS_SQL = """
select
    count(distinct {actor}::text) as actors,
    count(*) as total_rows,
    count(*) filter (where {ok}) as successful_rows,
    coalesce(
        sum({value}) filter (where {ok}),
        0
    ) as successful_value
from prime.{table}
where {segment_column} = :segment
  and {moment} >= cast(:period_start as date)
  and {moment} < cast(:period_end as date) + 1
"""

all_stats = query(
    ALL_STATS_SQL.format(
        table=TABLE,
        **COLUMNS[TABLE]
    ),
    segment=SEGMENT,
    period_start=PERIOD_START,
    period_end=PERIOD_END
)[0]

all_actors = int(all_stats["actors"])
all_rows = int(all_stats["total_rows"])
all_successful_rows = int(all_stats["successful_rows"])

all_ok_share = (
    100 * all_successful_rows / all_rows
)

all_value_per_actor = (
    float(all_stats["successful_value"])
    / all_actors
)


# Метрики по твоей тысяче
your_metrics = calculate_metrics(rows)


# Создаём 20 других случайных тысяч
salted_sql = SLICE_SQL.replace(
    "md5(actor_id)",
    "md5(actor_id || :salt)"
).format(
    table=TABLE,
    **COLUMNS[TABLE]
)


other_metrics = []

for number in range(1, 21):
    other_rows = query(
        salted_sql,
        segment=SEGMENT,
        period_start=PERIOD_START,
        period_end=PERIOD_END,
        salt=str(number)
    )

    other_metrics.append(
        calculate_metrics(other_rows)
    )


# Разброс показателей среди 20 других выборок
other_value_per_actor = [
    metric["value_per_actor"]
    for metric in other_metrics
]

other_ok_share = [
    metric["ok_share"]
    for metric in other_metrics
]


# Разница между твоей выборкой и всем сегментом
value_difference = round(
    100
    * (your_metrics["value_per_actor"] - all_value_per_actor)
    / all_value_per_actor,
    1
)

ok_share_difference = round(
    your_metrics["ok_share"] - all_ok_share,
    1
)


# Разброс случайных выборок
value_average = sum(other_value_per_actor) / len(
    other_value_per_actor
)

value_std = pstdev(other_value_per_actor)

ok_share_average = sum(other_ok_share) / len(
    other_ok_share
)

ok_share_std = pstdev(other_ok_share)


# Статистика по value на одного клиента во всём сегменте
ACTOR_VALUE_SQL = """
with actor_values as (
    select
        {actor}::text as actor_id,
        coalesce(
            sum({value}) filter (where {ok}),
            0
        ) as value_per_actor
    from prime.{table}
    where {segment_column} = :segment
      and {moment} >= cast(:period_start as date)
      and {moment} < cast(:period_end as date) + 1
    group by {actor}::text
)
select
    avg(value_per_actor) as average_value,
    stddev_pop(value_per_actor) as standard_deviation
from actor_values
"""

actor_stats = query(
    ACTOR_VALUE_SQL.format(
        table=TABLE,
        **COLUMNS[TABLE]
    ),
    segment=SEGMENT,
    period_start=PERIOD_START,
    period_end=PERIOD_END
)[0]

average_value = float(actor_stats["average_value"])
standard_deviation = float(
    actor_stats["standard_deviation"]
)


# Оценка необходимого размера выборки
# Используем ориентир 95% и падение на 5%
drop_value = 0.05 * average_value

if drop_value > 0:
    required_actors = ceil(
        (1.96 * standard_deviation / drop_value) ** 2
    )
else:
    required_actors = None


print("ВСЕГДА СЕГМЕНТ")
print("участников:", all_actors)
print("строк:", all_rows)
print(
    "доля успешных строк, %:",
    round(all_ok_share, 1)
)
print(
    "value на одного участника:",
    round(all_value_per_actor, 2)
)

print("\nТВОЯ ВЫБОРКА")
print("участников:", your_metrics["actors"])
print("строк:", your_metrics["rows"])
print(
    "доля успешных строк, %:",
    round(your_metrics["ok_share"], 1)
)
print(
    "value на одного участника:",
    round(your_metrics["value_per_actor"], 2)
)

print("\nОТЛИЧИЕ ОТ ВСЕГО СЕГМЕНТА")
print(
    "разница value на участника, %:",
    value_difference
)
print(
    "разница доли успешных строк, п.п.:",
    ok_share_difference
)

print("\n20 ДРУГИХ ВЫБОРОК")
print(
    "среднее value на участника:",
    round(value_average, 2)
)
print(
    "разброс value на участника:",
    round(value_std, 2)
)
print(
    "минимальное значение:",
    round(min(other_value_per_actor), 2)
)
print(
    "максимальное значение:",
    round(max(other_value_per_actor), 2)
)
print(
    "средняя доля успешных строк, %:",
    round(ok_share_average, 1)
)
print(
    "разброс доли успешных строк, п.п.:",
    round(ok_share_std, 1)
)

print("\nОЦЕНКА РАЗМЕРА ВЫБОРКИ")
print(
    "примерно нужно участников для обнаружения падения на 5%:",
    required_actors
)

#### ✍️ Исследование А
Во всём сегменте Value на одного участника составил 2 018,06, а в моей выборке — 1 928,46, то есть показатель оказался ниже на 4,4%; доля успешных операций отличается всего на 0,5 процентного пункта. В 20 других выборках Value на участника находился в диапазоне от 1 824,04 до 2 214,70, поэтому результат моей тысячи не выглядит аутлаером / ошибкой. При использованном расчёте для надёжного обнаружения падения показателя на 5% нужно примерно 3 080 участников, поэтому выборки из 1 000 клиентов для такой задачи недостаточно. Этот расчёт показывает разброс выборки, но не доказывает, что причины различий связаны с поведением клиентов, а не со случайным отбором.

### Исследование Б. Ваш сегмент особенный?

Маркетинг заметил, что в одних сегментах участники приносят заметно больше, чем в других, и хочет перераспределить бюджет в пользу «ценных» сегментов. Руководитель просит проверить: правда ли ваш сегмент отличается от соседних — и чем именно.

**Что сделать**

1. Для **всех сегментов** вашей таблицы за ваше окно посчитайте: число участников, строк на участника, долю строк с `ok = True`, `value` на одну строку с `ok = True` и `value` на участника.
2. Разложите `value` на участника на множители и найдите, какой из них даёт различие между сегментами.
3. Проверьте, держится ли картина в предыдущем окне той же длины. Данные начинаются 6 января 2025 года: если предыдущее окно начинается раньше, оно неполное — тогда сравнивайте доли и значения на участника или на строку, а не число участников и строк.
4. Вывод для маркетинга: «ценный» ли ваш сегмент; что на самом деле различается; какое решение по бюджету из этого следует — и чего ваши данные не доказывают.

**Подсказка.** Здесь нужны не выгрузка `rows`, а агрегаты по всей таблице за окно с разбивкой `group by` колонке сегмента.

In [ ]:
from datetime import date, timedelta


GROUP_STATS_SQL = """
select
    t.{segment_column}::text as segment,
    count(distinct t.{actor}::text) as actors,
    count(*) as total_rows,
    count(*) filter (where {ok}) as successful_rows,
    coalesce(
        sum({value}) filter (where {ok}),
        0
    ) as successful_value
from prime.{table} t
where t.{moment} >= cast(:period_start as date)
  and t.{moment} < cast(:period_end as date) + 1
group by t.{segment_column}
order by t.{segment_column}
"""


def get_segment_metrics(period_start, period_end):
    data = query(
        GROUP_STATS_SQL.format(
            table=TABLE,
            **COLUMNS[TABLE]
        ),
        period_start=period_start,
        period_end=period_end
    )

    metrics = []

    for row in data:
        actors = int(row["actors"])
        total_rows = int(row["total_rows"])
        successful_rows = int(row["successful_rows"])
        successful_value = float(row["successful_value"])

        rows_per_actor = total_rows / actors
        ok_share = successful_rows / total_rows

        if successful_rows > 0:
            value_per_successful_row = (
                successful_value / successful_rows
            )
        else:
            value_per_successful_row = 0

        value_per_actor = successful_value / actors

        metrics.append(
            {
                "segment": row["segment"],
                "actors": actors,
                "rows": total_rows,
                "rows_per_actor": rows_per_actor,
                "ok_share": ok_share,
                "value_per_successful_row": value_per_successful_row,
                "value_per_actor": value_per_actor,
            }
        )

    return metrics


# Текущее окно
current_metrics = get_segment_metrics(
    PERIOD_START,
    PERIOD_END
)


# Предыдущее окно такой же длины
current_start = date.fromisoformat(PERIOD_START)
current_end = date.fromisoformat(PERIOD_END)

window_days = (
    current_end - current_start
).days + 1

previous_end = current_start - timedelta(days=1)
previous_start = previous_end - timedelta(
    days=window_days - 1
)

previous_metrics = get_segment_metrics(
    previous_start.isoformat(),
    previous_end.isoformat()
)


# Печать показателей по сегментам
def print_metrics(title, metrics):
    print("\n" + title)

    for metric in metrics:
        print(
            metric["segment"],
            "| участников:",
            metric["actors"],
            "| строк на участника:",
            round(metric["rows_per_actor"], 2),
            "| успешных, %:",
            round(metric["ok_share"] * 100, 1),
            "| value на успешную строку:",
            round(metric["value_per_successful_row"], 2),
            "| value на участника:",
            round(metric["value_per_actor"], 2)
        )


print_metrics(
    "ТЕКУЩЕЕ ОКНО",
    current_metrics
)

print_metrics(
    "ПРЕДЫДУЩЕЕ ОКНО",
    previous_metrics
)


# Показатели твоего сегмента
your_segment = [
    metric
    for metric in current_metrics
    if metric["segment"] == SEGMENT
][0]

print("\nТВОЙ СЕГМЕНТ:", SEGMENT)
print(
    "строк на участника:",
    round(your_segment["rows_per_actor"], 2)
)
print(
    "доля успешных операций, %:",
    round(your_segment["ok_share"] * 100, 1)
)
print(
    "value на успешную строку:",
    round(your_segment["value_per_successful_row"], 2)
)
print(
    "value на участника:",
    round(your_segment["value_per_actor"], 2)
)


# Разложение value на участника
print("\nРАЗЛОЖЕНИЕ VALUE НА УЧАСТНИКА")

print(
    "строк на участника:",
    round(your_segment["rows_per_actor"], 2)
)

print(
    "доля успешных операций:",
    round(your_segment["ok_share"], 4)
)

print(
    "value на успешную строку:",
    round(
        your_segment["value_per_successful_row"],
        2
    )
)

print(
    "проверка произведения:",
    round(
        your_segment["rows_per_actor"]
        * your_segment["ok_share"]
        * your_segment["value_per_successful_row"],
        2
    )
)

print(
    "value на участника:",
    round(your_segment["value_per_actor"], 2)
)


# Кто лидирует по каждому показателю
print("\nЛИДЕРЫ ПО ПОКАЗАТЕЛЯМ")

leader_metrics = {
    "строк на участника": "rows_per_actor",
    "доля успешных операций": "ok_share",
    "value на успешную строку": "value_per_successful_row",
    "value на участника": "value_per_actor",
}

for metric_name, metric_key in leader_metrics.items():
    leader = max(
        current_metrics,
        key=lambda metric: metric[metric_key]
    )

    leader_value = leader[metric_key]

    if metric_key == "ok_share":
        leader_value = round(leader_value * 100, 1)
    else:
        leader_value = round(leader_value, 2)

    print(
        metric_name + ":",
        leader["segment"],
        "(",
        leader_value,
        ")"
    )


# Сравнение текущего и предыдущего окна
print("\nСРАВНЕНИЕ С ПРЕДЫДУЩИМ ОКНОМ")

previous_by_segment = {
    metric["segment"]: metric
    for metric in previous_metrics
}

for current_metric in current_metrics:
    segment = current_metric["segment"]
    previous_metric = previous_by_segment[segment]

    current_value = current_metric["value_per_actor"]
    previous_value = previous_metric["value_per_actor"]

    value_change = round(
        100 * (current_value - previous_value)
        / previous_value,
        1
    )

    current_ok = current_metric["ok_share"] * 100
    previous_ok = previous_metric["ok_share"] * 100

    print(
        segment,
        "| изменение value на участника, %:",
        value_change,
        "| изменение доли успешных, п.п.:",
        round(current_ok - previous_ok, 1)
    )

#### ✍️ Исследование Б

Я сравнила каналы app, pos и web по нескольким показателям. В текущем окне наибольшее value на одного участника у app — 2 018,06, тогда как у web этот показатель равен 1 653,14, а у pos — 1 541,67. Преимущество app связано не с более высокой суммой одной покупки: наоборот, самая высокая сумма на успешную операцию у pos — 1 369,66, а доля успешных операций у каналов почти одинаковая. Основной фактор — количество строк на одного участника: у app их 1,55 против 1,28 у web и 1,16 у pos, то есть клиенты в app чаще совершают покупки.
В предыдущем окне app также оставался лидером по value на участника — 2 012,96; за текущий период показатель вырос на 0,3%. Поэтому я бы рассматривала app как наиболее привлекательный канал для привлечения и удержания клиентов, но не стала бы сразу перераспределять весь бюджет только на основании этого расчёта. Данные показывают различия между каналами, но не доказывают, что именно канал является причиной более высокой ценности клиента: здесь не учитываются расходы на маркетинг, маржинальность и долгосрочное удержание клиентов.

## Напоследок

Что вас удивило в вашем участке? Одна-две фразы в ячейке ниже — по желанию и без баллов.

И пройдите, пожалуйста, **[короткий опрос о том, как вам это задание](https://forms.gle/5tNT4P4diHVyzhdFA)** — пара минут. Следующие задания соберу с учётом ответов.

#### ✍️ Что удивило

Меня удивило, что сегмент приложение оказался самым ценным сегментом не потому, что там самая большая сумма одной покупки, — а потому что на одного клиента приходится больше операций (видимо, приложение удобнее всег для ежедневгых / рутинных покупок). Ещё стало понятно, что выборка из 1 000 клиентов может заметно отличаться от всего сегмента: Value на участника оказался ниже на 4,4%, а для обнаружения падения на 5% нужна выборка примерно из 3 080 клиентов. Отдельно удивило, что ни после одной из 57 неуспешных операций не нашлось в тот же день повторной успешной операции того же клиента на ту же сумму: в 19 случаях после отмененной операции позже была успешная операция, но уже на другую сумму. Возможно, причина в том, что неуспешная операция чаще была не временным сбоем, который клиент сразу пытался повторить, а отклонённой или отменённой покупкой: после этого клиент либо не возвращался, либо совершал уже другую покупку (желательно отдельно проанализировать такие операции, возможно клиенты от нас уходят / есть тех проблемы с их способами оплаты / случа2но нажима.тся кнопки оплаты (проблемы с интерфейсом)).